# 05 · Каскад, сравнение, устойчивость, ошибки
Все модели оцениваются одной функцией на одинаковых test и held-out. Референс — готовая модель protectai.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

In [ ]:
from pig.pipeline import load_data, load_models, build_cascade, evaluate_all, ablation_table
from pig.eval.metrics import summary_table
splits = load_data(cfg)
models = load_models(cfg, 'sentence', with_transformer=True, with_reference=False)
print(list(models))

## 5.1 Каскад M4

In [ ]:
cas, info = build_cascade(cfg, models['M2_semantic'], models['M3_transformer'], splits['val'])
models[cas.name] = cas
info

## 5.2 Референс

In [ ]:
from pig.models.reference import ReferenceDetector
ref = ReferenceDetector(cfg['models']['reference']['name'])
models[ref.name] = ref

## 5.3–5.6 Оценка всех моделей
Порог референса подбирается на val тем же правилом FPR ≤ 2%.

In [ ]:
res = evaluate_all(cfg, models, splits, latency_n=200, val=splits['val'])
print('TEST'); display(summary_table(res['test']))
print('HELD-OUT'); display(summary_table(res['heldout']))
if 'manual' in res:
    print('MANUAL (ручной тест-сет)'); display(summary_table(res['manual']))
display(ablation_table(res).round(4))

In [ ]:
import pandas as pd
cols = ['model'] + [f'recall_{t}' for t in ['override', 'leak', 'roleplay', 'obfuscation']] + ['f1_ru', 'f1_en']
pd.DataFrame(res['test'])[cols].round(3)

In [ ]:
rob = pd.read_csv(cfg.path('reports') / 'robustness.csv')
rob.pivot_table(index='obfuscation', columns='model', values='recall').round(3)

In [ ]:
from IPython.display import Image
for f in ['curves_test.png', 'curves_heldout.png', 'recall_by_type.png', 'robustness.png']:
    display(Image(str(cfg.path('reports') / f)))

## Разбор ошибок M2/M4
Заполните колонку `reason` вручную (20–30 строк) и перенесите выводы в `experiments.md`.

In [ ]:
err = pd.read_csv(cfg.path('reports') / 'errors' / 'M2_semantic.csv')
pd.set_option('display.max_colwidth', 200)
err